In [5]:
# ==============================================================================
# CELL: INTERACTIVE GRADIO USER DEMO
# ==============================================================================
import gradio as gr
import torch
import numpy as np
from transformers import BertTokenizer, BertForSequenceClassification

print("--- LOADING SAVED MODELS FOR LIVE INTERACTIVE DEMO ---")

# 1. Load the fine-tuned Sentiment model and tokenizer
sentiment_model_path = "D:\\sentiment and sarcasm analysis\\models\\bert_sentiment"
sentiment_tokenizer = BertTokenizer.from_pretrained(sentiment_model_path)
sentiment_model = BertForSequenceClassification.from_pretrained(sentiment_model_path)
sentiment_model.eval()

# 2. Load the fine-tuned Sarcasm model and tokenizer
sarcasm_model_path = "D:\\sentiment and sarcasm analysis\\models\\bert_sarcasm"
sarcasm_tokenizer = BertTokenizer.from_pretrained(sarcasm_model_path)
sarcasm_model = BertForSequenceClassification.from_pretrained(sarcasm_model_path)
sarcasm_model.eval()

# Move both running models to your GPU to keep inference instant
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
sentiment_model.to(device)
sarcasm_model.to(device)

# 3. Core Prediction Function
def analyze_text(user_input):
    if not user_input.strip():
        return "Please enter some text to analyze.", "N/A"
    
    with torch.no_grad():
        # --- Task A: Sentiment Analysis ---
        sent_inputs = sentiment_tokenizer(
            user_input, max_length=512, padding="max_length", truncation=True, return_tensors="pt"
        ).to(device)
        sent_outputs = sentiment_model(**sent_inputs)
        sent_class = torch.argmax(sent_outputs.logits, dim=-1).item()
        sentiment_result = "POSITIVE" if sent_class == 1 else "NEGATIVE"
        
        # --- Task B: Sarcasm Detection ---
        sarc_inputs = sarcasm_tokenizer(
            user_input, max_length=64, padding="max_length", truncation=True, return_tensors="pt"
        ).to(device)
        sarc_outputs = sarcasm_model(**sarc_inputs)
        sarc_class = torch.argmax(sarc_outputs.logits, dim=-1).item()
        sarcasm_result = "SARCASTIC" if sarc_class == 1 else "NOT SARCASTIC"
        
    return sentiment_result, sarcasm_result

# 4. Build the custom Gradio UI layout
print("LAUNCHING GRADIO INTERFACE...")
demo = gr.Interface(
    fn=analyze_text,
    inputs=gr.Textbox(
        lines=3, 
        placeholder="Type a review or headline here... (e.g., 'Oh great, another broken feature, exactly what I wanted.')", 
        label="Input Text"
    ),
    outputs=[
        gr.Textbox(label="Predicted Sentiment Analysis"),
        gr.Textbox(label="Predicted Sarcasm Detection")
    ],
    title="Multi-Task NLP Analysis Portal",
    description="An introductory AI project comparing fine-tuned BERT models on Sentiment Analysis (IMDB Movie Reviews) and Sarcasm Detection (News Headlines).",
    examples=[
        ["This movie was absolute perfection! The acting and script were incredible."],
        ["What a masterpiece. I loved sitting in a broken chair for two hours listening to terrible dialogue."],
        ["Scientists discover new planet orbiting distant star system."]
    ],
    flagging_mode="never"
)

# Launch the local web server inside your notebook session
demo.launch(inline=True)


--- LOADING SAVED MODELS FOR LIVE INTERACTIVE DEMO ---


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 10014.08it/s]


LAUNCHING GRADIO INTERFACE...
* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
